# Source catalog inspection only

The catalog-sequence proposal was superseded by waveform-based EEW. This notebook only inspects the original CSV and caches its original values for trace matching and supervision; no historical sequence dataset, scaler or split is constructed. Origin-time ordering is retrospective and reporting availability is UNVERIFIED. Continue with `phase1_eew_dataset_preparation.ipynb`.


In [1]:
from pathlib import Path
import json,gzip,hashlib,sys,platform
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated": ROOT=ROOT/"bilstm new updated"
ROOT.mkdir(exist_ok=True)
SOURCE=ROOT.parent/"dataset_2.csv"
for name in (".cache","reports","data","research"): (ROOT/name).mkdir(exist_ok=True)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def save_json(path,value):
    p=Path(path); tmp=p.with_suffix(".tmp")
    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding="utf-8");tmp.replace(p)
SOURCE_HASH=sha(SOURCE)
cache=ROOT/".cache/source_snapshot.json.gz"
if cache.exists():
    with gzip.open(cache,"rt",encoding="utf-8") as f: cached=json.load(f)
    assert cached["sha256"]==SOURCE_HASH,"Source changed: preserve old outputs before starting a new version."
    raw=pd.DataFrame(cached["records"],columns=cached["columns"]).astype("string")
else:
    raw=pd.read_csv(SOURCE,dtype="string",keep_default_na=False)
    with gzip.open(cache,"wt",encoding="utf-8") as f:
        json.dump({"sha256":SOURCE_HASH,"columns":list(raw.columns),"records":raw.to_dict("records")},f)
source_row=np.arange(len(raw))
time=pd.to_datetime(raw.origin_time_utc.str.replace(" UTC","",regex=False),format="mixed",errors="coerce",utc=True)
num={c:pd.to_numeric(raw[c],errors="coerce").to_numpy(dtype=float) for c in
     ("Magnitude_Value","source_magnitude","Depth_km","source_depth_km","Lat","Long","source_latitude","source_longitude")}
types=raw.Magnitude_Type.str.strip().str.upper()
source_types=raw.source_magnitude_type.str.strip().str.upper()
both=np.isfinite(num["Magnitude_Value"])&np.isfinite(num["source_magnitude"])
delta=np.abs(num["Magnitude_Value"]-num["source_magnitude"])
summary={
    "source_path":str(SOURCE),"sha256":SOURCE_HASH,"rows":len(raw),"columns":list(raw.columns),
    "column_count":len(raw.columns),"missing_or_blank":{c:int((raw[c].isna()|raw[c].str.strip().eq("")).sum()) for c in raw},
    "valid_timestamp_rows":int(time.notna().sum()),"timestamp_min":time.min().isoformat(),"timestamp_max":time.max().isoformat(),
    "unique_source_ids":int(raw.source_id.nunique()),"duplicate_source_id_rows":int(raw.source_id.duplicated(keep=False).sum()),
    "exact_duplicate_rows":int(raw.duplicated().sum()),"tied_timestamp_rows":int(time.duplicated(keep=False).sum()),
    "magnitude_types":{str(k):int(v) for k,v in types.value_counts(dropna=False).items()},
    "source_magnitude_types":{str(k):int(v) for k,v in source_types.value_counts(dropna=False).items()},
    "magnitude_type_disagreements":int((types!=source_types).sum()),
    "magnitude_value_disagreements_gt_1e_6":int((both&(delta>1e-6)).sum()),
    "magnitude_value_difference_median":float(np.median(delta[both])),"magnitude_value_difference_max":float(delta[both].max()),
    "invalid_latitude":int((~np.isfinite(num["Lat"])|(np.abs(num["Lat"])>90)).sum()),
    "invalid_longitude":int((~np.isfinite(num["Long"])|(np.abs(num["Long"])>180)).sum()),
    "missing_or_negative_depth":int((~np.isfinite(num["Depth_km"])|(num["Depth_km"]<0)).sum()),
    "magnitude_by_type":{}
}
for typ in sorted(types.unique()):
    mask=types.eq(typ).to_numpy(dtype=bool)&np.isfinite(num["Magnitude_Value"])
    summary["magnitude_by_type"][typ]={"n":int(mask.sum()),"min":float(num["Magnitude_Value"][mask].min()),
        "max":float(num["Magnitude_Value"][mask].max()),"mean":float(num["Magnitude_Value"][mask].mean())}
save_json(ROOT/"reports/input_inspection.json",summary)
print(json.dumps({k:summary[k] for k in ("rows","column_count","valid_timestamp_rows","unique_source_ids","duplicate_source_id_rows",
    "exact_duplicate_rows","tied_timestamp_rows","magnitude_types","source_magnitude_types","magnitude_type_disagreements",
    "magnitude_value_disagreements_gt_1e_6","magnitude_value_difference_median","magnitude_value_difference_max",
    "invalid_latitude","invalid_longitude","missing_or_negative_depth","magnitude_by_type")},indent=2))


{
  "rows": 8089,
  "column_count": 28,
  "valid_timestamp_rows": 8089,
  "unique_source_ids": 8089,
  "duplicate_source_id_rows": 0,
  "exact_duplicate_rows": 0,
  "tied_timestamp_rows": 0,
  "magnitude_types": {
    "ML": 7125,
    "MB": 458,
    "MW": 398,
    "MC": 107,
    "MS": 1
  },
  "source_magnitude_types": {
    "ML": 7629,
    "MB": 298,
    "MC": 132,
    "MW": 30
  },
  "magnitude_type_disagreements": 553,
  "magnitude_value_disagreements_gt_1e_6": 540,
  "magnitude_value_difference_median": 0.0,
  "magnitude_value_difference_max": 2.1000000000000005,
  "invalid_latitude": 0,
  "invalid_longitude": 0,
  "missing_or_negative_depth": 0,
  "magnitude_by_type": {
    "MB": {
      "n": 458,
      "min": 3.9,
      "max": 6.6,
      "mean": 4.796943231441048
    },
    "MC": {
      "n": 107,
      "min": 1.7,
      "max": 4.6,
      "mean": 3.4046728971962623
    },
    "ML": {
      "n": 7125,
      "min": 1.5,
      "max": 8.6,
      "mean": 3.822091228070176
    },
    "M